# MorphoCoverage benchmark (P0-P5)

Runs the benchmark of `docs/common/validation.md` on Google Colab or locally.
The cells only call `data/simulations/decisions/colab.py`; every result is written to the output
folder, chunk by chunk, so a run cut by a Colab timeout continues where it stopped when cell 4 is
run again. Send back the zip of cell 5 (or the `summary.md` and `summary.csv` of each module).

Sizes: `smoke` (a test of the code, seconds), `pilot` (1% of the replicates: timing, memory and
convergence only, never a result), `full` (the protocol's counts, the confirmatory seed).

In [ ]:
# @title 1. Get the code
# On Colab: the repository is private, so it is cloned with a GitHub token stored in Colab's
# secrets as GITHUB_TOKEN (key icon on the left; read access to CellMigrationLab/MorphoCoverage).
# Without a token, upload a zip of the repository instead and set ZIP to its path.
import subprocess
import sys
from pathlib import Path

BRANCH = "main"
ZIP = ""  # e.g. "/content/MorphoCoverage.zip" when uploading the repository instead

if "google.colab" in sys.modules:
    REPO = Path("/content/MorphoCoverage")
    if not REPO.exists():
        if ZIP:
            subprocess.run(["unzip", "-q", ZIP, "-d", "/content"], check=True)
            REPO = next(Path("/content").glob("MorphoCoverage*/pyproject.toml")).parent
        else:
            from google.colab import userdata

            token = userdata.get("GITHUB_TOKEN")
            url = f"https://{token}@github.com/CellMigrationLab/MorphoCoverage.git"
            subprocess.run(
                ["git", "clone", "--depth", "1", "-b", BRANCH, url, str(REPO)], check=True
            )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[validation,dev]"], check=True
    )
else:
    REPO = next(
        p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/simulations/decisions").is_dir()
    )

for folder in (REPO / "src", REPO / "data/simulations"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

from decisions import colab

print("repository:", REPO)

In [ ]:
# @title 2. Output folder
# On Colab, Google Drive is mounted so the results survive the end of the session.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/morphocoverage_benchmark")
else:
    OUT = Path("decisions_results")
OUT.mkdir(parents=True, exist_ok=True)
print("results in:", OUT.resolve())

In [ ]:
# @title 3. What to run
# MODULES: any of "P0" (the deterministic tests), "P1", "P2", "P3", "P4", "P5".
# SIZE: "smoke", "pilot" or "full". WORKERS: None uses every CPU. CHUNK: replicates per file.
MODULES = ["P0", "P1", "P2", "P3", "P4", "P5"]
SIZE = "smoke"
WORKERS = None
CHUNK = 100

In [ ]:
# @title 4. Run (a progress line after each chunk, with the time left)
colab.run(MODULES, SIZE, OUT, WORKERS, CHUNK)

In [ ]:
# @title 5. Summarise and zip the results
import sys

colab.summarise(MODULES, OUT)
ARCHIVE = colab.archive(OUT)
print("send back:", ARCHIVE)
if "google.colab" in sys.modules:
    from google.colab import files

    files.download(str(ARCHIVE))